<a href="https://colab.research.google.com/github/UntalHugo/universidad-ia/blob/main/clase-01-rag-ventas/02_agente_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Agente RAG con Mistral + LangChain
Objetivo: crear un agente que consulte los datos de ventas y responda preguntas.

Celda 1 — Instalaciones


In [ ]:
!pip install pandas langchain langchain-experimental tabulate langchain-ollama langchain-mistralai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

Celda 2 — Configurar el Secret en Colab


Antes de escribir código, hay que cargar la API key. Seguí estos pasos:
1. En el panel izquierdo buscá el ícono de 🔑 llave
2. Clic en "Add new secret"
3. Completá así:
Name  → MISTRAL_API_KEY
Value → (pegás tu API key de Mistral acá)
4. Activá el toggle "Notebook access"

Celda 3 — Cargar la API key

In [ ]:
from google.colab import userdata
import os

os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
print("✅ API Key cargada correctamente")

✅ API Key cargada correctamente


Celda 4 — Testing de conexión con Mistral

In [ ]:
from langchain_mistralai import ChatMistralAI

# Inicializamos el modelo
# temperature=0 → respuestas deterministas, sin creatividad
# ideal para consultas sobre datos donde queremos precisión
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0
)

# Test simple para verificar que la conexión funciona
respuesta = llm.invoke("Respondé solo esto: ¿estás funcionando?")
print(respuesta.content)

Sí, estoy funcionando correctamente. ¿En qué puedo ayudarte?


Celda 5 — Cargar los datos normalizados


In [ ]:
import pandas as pd

# Cargamos las 4 tablas que normalizamos en el Colab 1
dim_clientes  = pd.read_csv("dim_clientes.csv")
dim_productos = pd.read_csv("dim_productos.csv")
fact_ordenes  = pd.read_csv("fact_ordenes.csv")
fact_items    = pd.read_csv("fact_items.csv")

print("✅ Tablas cargadas:")
print(f"   dim_clientes  → {dim_clientes.shape[0]} filas")
print(f"   dim_productos → {dim_productos.shape[0]} filas")
print(f"   fact_ordenes  → {fact_ordenes.shape[0]} filas")
print(f"   fact_items    → {fact_items.shape[0]} filas")

✅ Tablas cargadas:
   dim_clientes  → 92 filas
   dim_productos → 109 filas
   fact_ordenes  → 307 filas
   fact_items    → 2823 filas


Celda 6 — Importaciones


In [ ]:
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_mistralai import ChatMistralAI

# Ya no importamos AgentType, pasamos el string directamente
print("✅ Importaciones listas")

✅ Importaciones listas


Celda 7 — Crear el agente


In [ ]:
agente = create_pandas_dataframe_agent(
    llm=llm,
    df=[fact_items, fact_ordenes, dim_clientes, dim_productos],
    agent_type="tool-calling",    # ← cambiamos esto
    verbose=True,
    allow_dangerous_code=True
)

print("✅ Agente creado correctamente")

✅ Agente creado correctamente


Celda 8 — Primera pregunta al agente


In [ ]:
respuesta = agente.invoke("¿Cuantas ordenes hay en total?")
print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "total_orders = df1['ORDERNUMBER'].nunique()\ntotal_orders"}`


307Hay un total de **307 órdenes** en el dataframe `df1`.

> Finished chain.
Hay un total de **307 órdenes** en el dataframe `df1`.


Celda 9 — Preguntas de prueba


In [ ]:
preguntas = [
    "¿Cual es el producto mas vendido en cantidad?",
    "¿Que cliente genero mas ingresos en total?",
    "¿Cuantas ordenes fueron canceladas?",
]

for pregunta in preguntas:
    print(f"\n🔎 {pregunta}")
    print("-" * 50)
    respuesta = agente.invoke(pregunta)
    print(f"✅ {respuesta['output']}")
    print()


🔎 ¿Cual es el producto mas vendido en cantidad?
--------------------------------------------------


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Unir los dataframes necesarios para obtener la información de productos y cantidades vendidas\nmerged_df = df1.merge(df2, on='ORDERNUMBER').merge(df3, on='CUSTOMER_ID')\n\n# Agrupar por PRODUCTCODE y sumar las cantidades vendidas\nproduct_sales = merged_df.groupby('PRODUCTCODE')['QUANTITYORDERED'].sum()\n\n# Encontrar el producto con la mayor cantidad vendida\nmost_sold_product = product_sales.idxmax()\nmost_sold_quantity = product_sales.max()\n\nmost_sold_product, most_sold_quantity"}`


('S18_3232', 1774)El producto más vendido en cantidad es el **S18_3232** con un total de **1774** unidades vendidas.

> Finished chain.
✅ El producto más vendido en cantidad es el **S18_3232** con un total de **1774** unidades vendidas.


🔎 ¿Que cliente genero mas ingresos en total?
-----------

Celda 10 — Chat interactivo


In [ ]:
print("🤖 Agente de ventas listo. Escribí 'salir' para terminar.\n")

while True:
    pregunta = input("Vos: ")

    if pregunta.lower() == "salir":
        print("👋 Cerrando el agente.")
        break

    if pregunta.strip() == "":
        continue

    respuesta = agente.invoke(pregunta)
    print(f"\n🤖 Agente: {respuesta['output']}\n")

🤖 Agente de ventas listo. Escribí 'salir' para terminar.



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "clientes_con_nombre_hugo = [df3[df3['CUSTOMERNAME'].str.contains('Hugo', case=False, na=False)]]\nclientes_con_nombre_hugo"}`


[Empty DataFrame
Columns: [CUSTOMER_ID, CUSTOMERNAME, PHONE, ADDRESSLINE1, ADDRESSLINE2, CITY, STATE, POSTALCODE, COUNTRY, TERRITORY, CONTACTLASTNAME, CONTACTFIRSTNAME]
Index: []]No hay ningún cliente con el nombre "Hugo" en el dataframe `df3`.

> Finished chain.

🤖 Agente: No hay ningún cliente con el nombre "Hugo" en el dataframe `df3`.



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Identificamos el producto más caro en el dataframe df4 (PRODUCTCODE, PRODUCTLINE, MSRP)\ndf4_max_price = df4.loc[df4['MSRP'].idxmax()]\ndf4_max_price[['PRODUCTCODE', 'PRODUCTLINE', 'MSRP']]"}`


PRODUCTCODE        S10_1949
PRODUCTLINE    Classic Cars
MSRP                    214
Name: 1, dtype: obj

Celda 11 — Instalar ChromaDB


In [ ]:
!pip install chromadb langchain-chroma